# Online Retail Intelligence — EDA
V2 reconstruction. Run `python src/pipeline.py` from repo root before opening. Outputs are calculated from your local data; not historical report values.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
root = Path.cwd() if (Path.cwd()/'data').is_dir() else Path.cwd().parent
p = root/'data'/'processed'
fact = pd.read_csv(p/'FactSales.csv', dtype={'InvoiceNo':str,'StockCode':str,'CustomerID':str}, parse_dates=['Date'])
customer = pd.read_csv(p/'DimCustomer.csv', dtype={'CustomerID':str})
product = pd.read_csv(p/'DimProduct.csv', dtype={'StockCode':str})
revenue=fact.Revenue.sum(); orders=fact.InvoiceNo.nunique()
print({'Revenue GBP':revenue,'Orders':orders,'Customers':fact.CustomerID.nunique(),'AOV GBP':revenue/orders})

In [ ]:
monthly=fact.assign(Month=fact.Date.dt.to_period('M').astype(str)).groupby('Month',as_index=False).Revenue.sum()
px.line(monthly,x='Month',y='Revenue',markers=True,title='Monthly sales revenue (GBP)').show()

In [ ]:
market=fact.groupby('Country',as_index=False).Revenue.sum().nlargest(10,'Revenue')
px.bar(market,x='Revenue',y='Country',orientation='h',title='Top 10 customer markets by sales').show()

In [ ]:
top=fact.groupby('StockCode',as_index=False).Revenue.sum().nlargest(10,'Revenue').merge(product,on='StockCode')
px.bar(top,x='Revenue',y='StockCode',hover_data=['Description'],orientation='h',title='Top 10 SKUs by sales').show()

In [ ]:
counts=customer.Segment.value_counts().rename_axis('Segment').reset_index(name='Customers')
px.pie(counts,names='Segment',values='Customers',hole=.5,title='Full-period RFM customer snapshot').show()